# Tokenization
How text becomes tokens — BPE algorithm, vocabulary construction, and why tokenization matters for LLM behavior.

## 1. Theory & Intuition

### What is a Token?
A token is NOT a word — it is a subword unit that appears frequently enough in training data to get its own ID.

**Three approaches:**
| Approach | Example | Vocab Size | Problem |
|----------|---------|------------|---------|
| Character level | cat = [c,a,t] | ~256 | Too many tokens per word |
| Word level | cat = [cat] | 170,000+ | Cannot handle rare/new words |
| Subword (BPE) | cat = [cat] | ~50,000 | Best of both worlds |

**Why ' the' is not equal to 'the':**
Space is part of the token. 'the' at word start vs middle = different frequencies = different tokens.
This is why GPT tokenizers use a special character to mark word boundaries.

### BPE — Byte Pair Encoding Algorithm
Used by GPT, Llama, Mistral, and most modern LLMs.

**Steps:**
1. Start with character-level vocabulary
2. Count all adjacent character pairs in training corpus
3. Merge the most frequent pair into a new token
4. Repeat until vocabulary reaches target size (e.g. 50,000 tokens)

**Key insight:** BPE learns morphology automatically — common words become single tokens, rare words split into meaningful subword pieces.

### Why Tokenization Matters for Interviews
- Token count affects cost and context limits
- Numbers tokenize badly — '1234' may be 3-4 tokens
- Non-English text uses more tokens per word
- Code tokenizes differently than prose
- Prompt length in tokens != prompt length in characters

## 2. Implementation — BPE From Scratch

In [ ]:
from collections import defaultdict

def get_vocab(corpus):
    vocab = defaultdict(int)
    for word in corpus.split():
        chars = ' '.join(list('G' + word))
        vocab[chars] += 1
    return vocab

def get_pairs(vocab):
    pairs = defaultdict(int)
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[(symbols[i], symbols[i+1])] += freq
    return pairs

def merge_vocab(pair, vocab):
    new_vocab = {}
    bigram = ' '.join(pair)
    replacement = ''.join(pair)
    for word in vocab:
        new_word = word.replace(bigram, replacement)
        new_vocab[new_word] = vocab[word]
    return new_vocab

corpus = 'low low low lower lower newest newest highest'
vocab = get_vocab(corpus)

print('Initial vocabulary:')
for word, freq in vocab.items():
    print(f'  {word!r} : {freq}')

merges = []
for i in range(10):
    pairs = get_pairs(vocab)
    if not pairs:
        break
    best_pair = max(pairs, key=pairs.get)
    vocab = merge_vocab(best_pair, vocab)
    merges.append(best_pair)
    print(f'Merge {i+1}: {best_pair[0]!r} + {best_pair[1]!r} -> {chr(39).join(best_pair)!r}')

print('\nFinal learned merges:')
for i, merge in enumerate(merges):
    print(f'  {i+1}. {merge[0]} + {merge[1]} -> {chr(39).join(merge)}')

## 3. Hands-on Practice — HuggingFace Tokenizers

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'transformers', '-q'], capture_output=True)

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('gpt2')

test_texts = [
    'Hello world',
    'tokenization',
    'ChatGPT is amazing',
    '1234567890',
    'def fibonacci(n): return n if n <= 1 else fibonacci(n-1) + fibonacci(n-2)',
]

print('GPT-2 Tokenization Analysis')
print('='*60)
for text in test_texts:
    tokens = tokenizer.encode(text)
    token_strings = tokenizer.convert_ids_to_tokens(tokens)
    print(f'\nText: {text!r}')
    print(f'Tokens ({len(tokens)}): {token_strings}')

In [ ]:
texts = {
    'Short sentence': 'The cat sat on the mat.',
    'Technical text': 'Implement a binary search algorithm with O(log n) time complexity.',
    'Code snippet': 'for i in range(len(nums)): if nums[i] == target: return i',
    'Numbers': 'The price is $1,234.56 and the date is 2024-01-15',
}

print('Token Count Analysis')
print('='*60)
for name, text in texts.items():
    tokens = tokenizer.encode(text)
    ratio = len(text) / len(tokens)
    print(f'\n{name}:')
    print(f'  Characters: {len(text)}')
    print(f'  Tokens: {len(tokens)}')
    print(f'  Chars per token: {ratio:.1f}')

print('\nKey insight: ~4 chars per token for English prose')
print('Numbers, code, and non-English text are less efficient')

## 4. Production Application

In [ ]:
print('Production Tokenization Patterns')
print('='*60)

def count_tokens(text, tokenizer):
    return len(tokenizer.encode(text))

prompt = 'Explain the transformer architecture in detail with examples.'
token_count = count_tokens(prompt, tokenizer)
print(f'Prompt token count: {token_count}')
print(f'Estimated cost at $0.01/1K tokens: ${token_count * 0.00001:.6f}')

def truncate_to_limit(text, tokenizer, max_tokens=50):
    tokens = tokenizer.encode(text)
    if len(tokens) > max_tokens:
        tokens = tokens[:max_tokens]
        return tokenizer.decode(tokens), True
    return text, False

long_text = 'The transformer architecture ' * 20
truncated, was_truncated = truncate_to_limit(long_text, tokenizer)
print(f'Was truncated: {was_truncated}')
print(f'Truncated preview: {truncated[:80]}...')

## 5. Key Takeaways

### Summary
| Concept | Key Point |
|---------|----------|
| Token | Subword unit, NOT a word |
| BPE | Iteratively merges most frequent character pairs |
| Word boundary | Space is part of the token — ' the' is not equal to 'the' |
| Vocab size | ~50,000 for most modern LLMs |
| Efficiency | ~4 chars/token for English, less for numbers/code |

### Key Points
1. BPE learns morphology from frequency — no linguistic rules needed
2. Common words get single tokens, rare words split into subword pieces
3. Numbers tokenize poorly — '1234' can be 3-4 separate tokens
4. Always count tokens before API calls — affects cost and context limits
5. Special tokens ([BOS], [EOS], chat format tokens) add to your count
6. Non-English text uses more tokens per word than English

### Interview Question You Should Answer Cold
**Q: How does BPE tokenization work and why does GPT tokenize ' the' differently from 'the'?**

A: BPE iteratively merges the most frequent adjacent character pairs in training data until reaching target vocabulary size. ' the' and 'the' appear in different positions with different frequencies, so they get merged into separate tokens. The space is part of the token, not a separator.